In [1]:
import os
import copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import matplotlib.cm as cm

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
# ============================================================
# ARQUITECTURA DEL CAE (sin cambios)
# ============================================================
 
class Conv1DAutoencoder(nn.Module):
 
    def __init__(self, input_channels=4, seq_len=61, bottleneck_dim=16):
        super().__init__()
 
        self.input_channels = input_channels
        self.seq_len = seq_len
 
        self.encoder = nn.Sequential(
            nn.Conv1d(input_channels, 64, kernel_size=7, stride=1, padding=1),
            nn.ELU(),
            nn.MaxPool1d(kernel_size=2, stride=2),
 
            nn.Conv1d(64, 128, kernel_size=5, stride=1, padding=0),
            nn.ELU(),
            nn.MaxPool1d(kernel_size=2, stride=2),
 
            nn.Conv1d(128, 256, kernel_size=3, stride=1, padding=0),
            nn.ELU(),
            nn.MaxPool1d(kernel_size=2, stride=2),
        )
 
        self.flatten = nn.Flatten()
 
        self.fc_enc = nn.Sequential(
            nn.Linear(256 * 5, 128), nn.ELU(),
            nn.Linear(128, 64), nn.ELU(),
            nn.Linear(64, 32), nn.ELU(),
            nn.Linear(32, bottleneck_dim),
        )
 
        self.fc_dec = nn.Sequential(
            nn.Linear(bottleneck_dim, 32), nn.ELU(),
            nn.Linear(32, 64), nn.ELU(),
            nn.Linear(64, 128), nn.ELU(),
            nn.Linear(128, input_channels * seq_len)
        )
 
    def forward(self, x):
        x_enc = self.encoder(x)
        x_flat = self.flatten(x_enc)
        z = self.fc_enc(x_flat)
        x_dec = self.fc_dec(z)
        return x_dec.view(-1, self.input_channels, self.seq_len)
 

In [ ]:
# ============================================================
# FUNCIÓN DE ENTRENAMIENTO POR CLASE
# (validación congelada, scheduler y mejor época por val)
# ============================================================
 
def entrenar_cae_por_clase(
    X_train,
    X_val,
    model_class,
    cultivo,
    ruta_modelo,
    input_channels,
    seq_len=61,
    bottleneck_dim=16,
    batch_size=128,
    epochs=20,
    lr=1e-3,
    device='cpu',
):
    loader = DataLoader(TensorDataset(X_train, X_train), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_val, X_val), batch_size=batch_size, shuffle=False)
 
    model = model_class(input_channels=input_channels, seq_len=seq_len, bottleneck_dim=bottleneck_dim).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
 
    mejor_loss = float('inf')
    mejor_weights = None
    mejor_epoch = 0
    hist_train, hist_val = [], []
 
    for epoch in range(epochs):
        # --- Entrenamiento ---
        model.train()
        loss_train = 0.0
        for x_batch, _ in loader:
            x_batch = x_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x_batch), x_batch)
            loss.backward()
            optimizer.step()
            loss_train += loss.item() * x_batch.size(0)
        loss_train /= len(loader.dataset)
 
        # --- Validación (no actualiza pesos) ---
        model.eval()
        loss_val = 0.0
        with torch.no_grad():
            for x_batch, _ in val_loader:
                x_batch = x_batch.to(device)
                loss = criterion(model(x_batch), x_batch)
                loss_val += loss.item() * x_batch.size(0)
        loss_val /= len(val_loader.dataset)
 
        hist_train.append(loss_train)
        hist_val.append(loss_val)
        print(f"Epoch {epoch+1}/{epochs} - {cultivo} - Train: {loss_train:.4f} - Val: {loss_val:.4f}")
 
        scheduler.step(loss_val)
        if loss_val < mejor_loss:
            mejor_loss = loss_val
            mejor_epoch = epoch
            mejor_weights = copy.deepcopy(model.state_dict())
 
    # Guardar el mejor modelo de la clase
    os.makedirs(os.path.dirname(ruta_modelo), exist_ok=True)
    torch.save({
        'model_state_dict': mejor_weights,
        'loss': mejor_loss,
        'epoch': mejor_epoch,
    }, ruta_modelo)
    print(f"🏆 {cultivo}: mejor época {mejor_epoch+1} (Val MSE: {mejor_loss:.6f})")
 
    model.load_state_dict(mejor_weights)
    return model, mejor_loss, mejor_epoch, hist_train, hist_val

In [ ]:
# ============================================================
# CARGA DEL DATASET
# ============================================================
 
data = torch.load(
    "/home/angelo/MyStore/Proyecto Investigador/dataset_split_2019/dataset_split_2019_(backscatter,coh).pt",
    weights_only=False
)
 
datos_train_por_cultivo = data["train"]
 
datos_limpios = {}
for clase, datos in datos_train_por_cultivo.items():
    X = torch.tensor(datos["X"]).float()
    X = X.reshape(X.shape[0], 4, 61)
    datos_limpios[clase] = {
        "X": X,
        "coords": torch.tensor(datos["coords"]),
        "parcelas": torch.tensor(datos["parcelas"])
    }
 
print("✅ Dataset cargado.")
for clase, datos in datos_limpios.items():
    print(f"  {clase}: X{tuple(datos['X'].shape)}")

✅ Dataset cargado.
  Alfalfa: X(6980, 2, 61)
  Barley: X(3027, 2, 61)
  Carrot: X(3508, 2, 61)
  Chickpea: X(1262, 2, 61)
  Cotton: X(271429, 2, 61)
  Fallow: X(6559, 2, 61)
  Maize: X(37004, 2, 61)
  Oats: X(3097, 2, 61)
  Onion: X(11875, 2, 61)
  Pepper: X(13484, 2, 61)
  Potato: X(1161, 2, 61)
  Pumpkin: X(3717, 2, 61)
  Quinoa: X(5279, 2, 61)
  Rice: X(112685, 2, 61)
  SugarBeet: X(48463, 2, 61)
  Sunflower: X(10658, 2, 61)
  SweetPotato: X(10323, 2, 61)
  Tomato: X(119534, 2, 61)
  Wheat: X(11506, 2, 61)


In [ ]:
# ============================================================
# CONFIGURACIÓN
# ============================================================
 
path_modelos = "/home/angelo/MyStore/Proyecto Investigador/Modelos_temporales_2019_200-16_coherencia_sin purga"
os.makedirs(path_modelos, exist_ok=True)
 
INPUT_CHANNELS = 4
SEQ_LEN = 61
BOTTLENECK_DIM = 16
EPOCHS = 200

BATCH_SIZE = 128
LR = 1e-3
 
val_frac = 0.15
seed_val_global = 42
 
# ============================================================
# SPLIT TRAIN/VAL (fijo, reproducible)
# ============================================================
 
datos_train = {}
datos_val = {}
 
for clase, datos_orig in datos_limpios.items():
    n_total = datos_orig["X"].shape[0]
    generador = torch.Generator().manual_seed(seed_val_global)
    n_val = max(1, int(n_total * val_frac))
    n_train = n_total - n_val
 
    idx_perm = torch.randperm(n_total, generator=generador)
    idx_train, idx_val = idx_perm[:n_train], idx_perm[n_train:]
 
    datos_train[clase] = {k: v[idx_train] for k, v in datos_orig.items()}
    datos_val[clase] = {k: v[idx_val] for k, v in datos_orig.items()}
 
print("✅ Split train/val fijo realizado.")
 
print("\n===== CONTROL DE DATOS =====")
for clase, datos in datos_train.items():
    X = datos["X"]
    print(clase, "shape:", tuple(X.shape), "finite:", torch.isfinite(X).all().item(),
          "min:", X.min().item(), "max:", X.max().item())
 
# %%
# ============================================================
# ENTRENAMIENTO ÚNICO (sin purga, una sola iteración)
# ============================================================
 
modelos = {}
resultados = {}
history_train = {}
history_val = {}
 
for clase in datos_train.keys():
    print(f"\n Entrenando CAE para clase {clase}...")
 
    model, mejor_loss, mejor_epoch, h_tr, h_va = entrenar_cae_por_clase(
        datos_train[clase]["X"],
        datos_val[clase]["X"],
        Conv1DAutoencoder,
        clase,
        ruta_modelo=f"{path_modelos}/mejor_modelo_{clase}.pth",
        input_channels=INPUT_CHANNELS,
        seq_len=SEQ_LEN,
        bottleneck_dim=BOTTLENECK_DIM,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        lr=LR,
        device=device,
    )
 
    modelos[clase] = model
    resultados[clase] = {"loss": mejor_loss, "epoch": mejor_epoch}
    history_train[clase] = h_tr
    history_val[clase] = h_va
 
# %%
# ============================================================
# 📊 GRÁFICO DE PÉRDIDA POR CLASE (por época)
# ============================================================
 
cultivos = list(history_val.keys())
colores = cm.tab20(np.linspace(0, 1, len(cultivos)))
 
plt.figure(figsize=(12, 6))
for color, clase in zip(colores, cultivos):
    plt.plot(range(1, len(history_train[clase]) + 1), history_train[clase],
             linestyle='--', color=color, alpha=0.6)
    plt.plot(range(1, len(history_val[clase]) + 1), history_val[clase],
             marker='o', color=color, label=f'Clase {clase}')
 
plt.xlabel("Época")
plt.ylabel("MSE")
plt.title("Evolución del MSE por clase (discontinua: train, continua: val)")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
plt.grid(True)
plt.tight_layout()
plt.savefig(f"{path_modelos}/evolucion_perdidas.png", bbox_inches='tight')
plt.show()
 
# %%
# ============================================================
# 🏆 RESUMEN FINAL
# ============================================================
 
print("\n" + "=" * 60)
print("RESUMEN FINAL DE MODELOS ENTRENADOS")
print("=" * 60)
 
for clase, r in resultados.items():
    print(f"Clase {clase}:")
    print(f"   Mejor época: {r['epoch'] + 1}")
    print(f"   Mejor pérdida val (MSE): {r['loss']:.6f}")
    print(f"   Archivo: mejor_modelo_{clase}.pth\n")
 
print("Entrenamiento completado.")
 

Utilizando dispositivo: cuda
Split Train/Val realizado a nivel de PARCELA.

PASADA DE PURGA 1/10

Entrenando CAE (Alfalfa) - Muestras actuales: torch.Size([4658, 2, 61])
Epoch 1/20 - Alfalfa - Train: 0.0568 - Val: 0.0119
Epoch 2/20 - Alfalfa - Train: 0.0121 - Val: 0.0106
Epoch 3/20 - Alfalfa - Train: 0.0114 - Val: 0.0104
Epoch 4/20 - Alfalfa - Train: 0.0113 - Val: 0.0106
Epoch 5/20 - Alfalfa - Train: 0.0113 - Val: 0.0103
Epoch 6/20 - Alfalfa - Train: 0.0112 - Val: 0.0103
Epoch 7/20 - Alfalfa - Train: 0.0110 - Val: 0.0104
Epoch 8/20 - Alfalfa - Train: 0.0108 - Val: 0.0104
Epoch 9/20 - Alfalfa - Train: 0.0105 - Val: 0.0105
Epoch 10/20 - Alfalfa - Train: 0.0103 - Val: 0.0105
Epoch 11/20 - Alfalfa - Train: 0.0102 - Val: 0.0106
Epoch 12/20 - Alfalfa - Train: 0.0102 - Val: 0.0106
Epoch 13/20 - Alfalfa - Train: 0.0101 - Val: 0.0106
Epoch 14/20 - Alfalfa - Train: 0.0101 - Val: 0.0105
Epoch 15/20 - Alfalfa - Train: 0.0101 - Val: 0.0106
Epoch 16/20 - Alfalfa - Train: 0.0100 - Val: 0.0108
Epoch 1

KeyboardInterrupt: 